# Dynex SDK - nBit Adder Native Gate Circuit Example

First we import the required packages:

In [4]:
from pennylane import numpy as np
import pennylane as qml
from dynex import DynexConfig, ComputeBackend, DynexCircuit

config = DynexConfig(compute_backend=ComputeBackend.QPU, qpu_model="apollo_rc1", use_notebook_output=True)

We define our circuit:

In [5]:
params = [28938284928182, 2722366482869645212131]  # two numbers to add


def Nqubits(a, b):
    mxVal = a + b
    return mxVal.bit_length()


wires = Nqubits(*params)


def Kfourier(k, wires):
    for j in range(len(wires)):
        qml.RZ(k * np.pi / (2 ** j), wires=wires[j])


def FullAdder(params, state=True):
    a, b = params
    wires = Nqubits(a, b)
    qml.BasisEmbedding(a, wires=range(wires))
    qml.QFT(wires=range(wires))
    Kfourier(b, range(wires))
    qml.adjoint(qml.QFT)(wires=range(wires))
    if state:
        return qml.state()
    else:
        return qml.sample()

We draw the circuit:

In [6]:
# draw circuit:
# too large to draw _ = qml.draw_mpl(FullAdder, style="black_white")(params)
print("Cant't draw circuit with", Nqubits(params[0], params[1]), "wires");

Cant't draw circuit with 72 wires


We execute and measure the circuit on the Dynex platform:

In [7]:
# Execute the circuit on Dynex:
dynex_circuit = DynexCircuit(config=config)
measure = dynex_circuit.execute(FullAdder, params, wires, method="measure",
                                num_reads=1, integration_steps=10)
print("Mesaure:", measure)

INFO: [DYNEX-APOLLO-RC1] Executing PennyLane quantum circuit
INFO: [DYNEX-APOLLO-RC1] Sampler initialised
INFO: [DYNEX-APOLLO-RC1] Apollo QPU chip: apollo_rc1
INFO: [DYNEX-APOLLO-RC1] Settings: num_reads=1, shots=1, annealing_time=10
INFO: [DYNEX-APOLLO-RC1] Submitting the job to Dynex.
INFO: [DYNEX-APOLLO-RC1] SUCCESS: Job created successfully (job_id=7414)
INFO: [DYNEX-APOLLO-RC1] feed_dict: {'cos_rz_0': 0.8488523815916276, 'cos_rz_1': 0.9614708476057992, 'cos_rz_10': -0.8873889876367417, 'cos_rz_11': -0.23728781296482365, 'cos_rz_12': 0.6175403578047253, 'cos_rz_13': 0.899316506521682, 'cos_rz_14': -0.9745041063334936, 'cos_rz_15': -0.11290680596515512, 'cos_rz_16': -0.6659929406663575, 'cos_rz_17': -0.4086606534361012, 'cos_rz_18': 0.5437551593152468, 'cos_rz_19': 0.8785656376490167, 'cos_rz_2': -0.9903208691140966, 'cos_rz_20': 0.9691660429588463, 'cos_rz_21': -0.9922615690831844, 'cos_rz_22': -0.062203018081181484, 'cos_rz_23': -0.6847616307587694, 'cos_rz_24': -0.397012826770893

Mesaure: [1 0 0 1 0 0 1 1 1 0 0 1 0 1 0 0 0 1 1 0 1 1 0 0 1 0 1 1 1 1 1 0 1 1 0 0 0
 1 0 1 1 1 1 1 1 0 1 0 1 1 1 1 0 0 0 1 1 0 0 0 0 1 1 0 1 0 0 1 1 0 0 1]


In [8]:
bitStr = "".join(map(str, measure.astype(int)))
dynexResult = int(bitStr, 2)
print("Dynex Result:", dynexResult)
print("Expected Result:", sum(params))
isValidDynex = dynexResult == sum(params)
print("Is Dynex Result Valid?", isValidDynex)

Dynex Result: 2722366511807930140313
Expected Result: 2722366511807930140313
Is Dynex Result Valid? True
